# Strands Decider Example

This example uses `DeciderBackend` with [Strands Decider](https://github.com/strands-labs/strands-decider), a 2B open-weights decision model. Decision models do not generate text: each question is answered with probabilities read off the model, so the answers cannot leave the schema and `confidence` is measured, not self-reported.

```bash
pip install grounded-ai[decider]
```

In [1]:
from grounded_ai import Evaluator
from grounded_ai.backends.decider import (
    HALLUCINATION,
    RAG_RELEVANCE,
    TOXICITY,
    ChoiceQuestion,
    DeciderInput,
    NoulQuestion,
    ScoreQuestion,
)

## Start the server

`warmup()` runs `strands-decider serve` on the port you give it (CUDA, Apple Silicon or CPU), waits until it is ready, and points the evaluator at it. The first start downloads the checkpoint. A server already running on that port is reused.

In [2]:
# Uses the 'decider/' prefix to trigger auto-detection; the rest is the checkpoint to serve
evaluator = Evaluator("decider/StrandsAgents/strands-decider-2B-hobson-v19")
evaluator.backend.warmup(port=8000)
print("Server ready at", evaluator.backend.base_url)

Server ready at http://127.0.0.1:8000


## Hallucination

Every call takes a `DeciderInput`: a `state` (what the model reads, as text or JSON) and the `questions` it is asked. The result is a `DeciderOutput`. `HALLUCINATION` is a ready-made question. The answer carries the pick, the probability of every option, and the confidence.

In [3]:
context = """
The Apollo 11 mission landed the first humans on the Moon.
Neil Armstrong and Buzz Aldrin walked on the lunar surface.
Michael Collins remained in orbit in the Command Module.
"""
query = "Who stayed in orbit?"

for response in [
    "Michael Collins remained in orbit.",  # accurate
    "Buzz Aldrin stayed in the orbiter while Neil went down alone.",  # inaccurate
]:
    result = evaluator.evaluate(DeciderInput(
        state={"context": context, "query": query, "response": response},
        questions={"verdict": HALLUCINATION},
    ))
    verdict = result.answers["verdict"]
    print(f"{verdict.choice:14s} confidence={verdict.confidence:.3f}  {verdict.probabilities}")

faithful       confidence=0.854  {'hallucination': 0.0729, 'faithful': 0.9271}
hallucination  confidence=0.324  {'hallucination': 0.662, 'faithful': 0.338}


In [4]:
# The other ready-made questions: TOXICITY and RAG_RELEVANCE
for text in ["You are a worthless idiot and everyone hates you.", "Thanks for the quick reply, that fixed it."]:
    tone = evaluator.evaluate(DeciderInput(state={"response": text}, questions={"tone": TOXICITY})).answers["tone"]
    print(f"{tone.choice:10s} confidence={tone.confidence:.3f}  {text}")

relevance = evaluator.evaluate(DeciderInput(
    state={
        "query": "What are the benefits of vitamin D?",
        "response": "The Eiffel Tower is located in Paris, France.",
    },
    questions={"relevance": RAG_RELEVANCE},
)).answers["relevance"]
print(relevance.choice, relevance.probabilities)

toxic      confidence=0.967  You are a worthless idiot and everyone hates you.
non-toxic  confidence=0.918  Thanks for the quick reply, that fixed it.
unrelated {'relevant': 0.1438, 'unrelated': 0.8562}


## Your own questions

There are three question types, and each has its own answer. Any number of questions go in one request.

| Question | Answer |
| :--- | :--- |
| `NoulQuestion` (a yes/no statement) | `.noul` |
| `ChoiceQuestion` (options with descriptions) | `.choice`, `.probabilities`, `.confidence` |
| `ScoreQuestion` (levels, lowest first) | `.score` (level index from 0), `.legend`, `.probabilities`, `.confidence` |

In [5]:
result = evaluator.evaluate(DeciderInput(
    state="You have charged me twice and my account is now overdrawn. I need this reversed today.",
    questions={
        "urgent": NoulQuestion(instructions="This needs a reply within the hour."),
        "area": ChoiceQuestion(
            instructions="Which team owns it?",
            criteria={
                "billing": "charges and refunds",
                "bug": "the product misbehaves",
                "account": "login and profile",
            },
        ),
        "clarity": ScoreQuestion(
            instructions="How clearly is the problem described?",
            criteria=["unclear", "partly clear", "clear"],
        ),
    },
))

for name, answer in result.answers.items():
    print(name, answer)

urgent type='noul' noul=0.5887
area type='choice' choice='billing' probabilities={'billing': 0.937, 'bug': 0.0351, 'account': 0.028} confidence=0.9054
clarity type='score' score=1.3147 legend={'0': 'unclear', '1': 'partly clear', '2': 'clear'} probabilities={'0': 0.1876, '1': 0.31, '2': 0.5024} confidence=0.3382


## Code Review: the same example as the Anthropic and OpenAI notebooks

Those notebooks pass a custom `CodeReviewInput` and a custom `CodeReviewOutput`. Decider has its own input and output classes, separate from `EvaluationInput` and `EvaluationOutput`. The input is still yours to shape, but **the output is not a schema you define**: it is the model's answers to the questions you ask.

| Anthropic / OpenAI notebook | Here |
| :--- | :--- |
| `class CodeReviewInput(BaseModel)` | `class CodeReviewInput(DeciderInput)`, same fields |
| `is_bug_free: bool` | a `NoulQuestion` -> `.noul` |
| `security_risk: bool` | a `NoulQuestion` -> `.noul` |
| `complexity_score: int` | a `ScoreQuestion` -> `.score`, `.probabilities`, `.confidence` |
| `suggestions: List[str]` | not possible: a decision model does not write text |
| `output_schema=CodeReviewOutput` | always a `DeciderOutput` |

In [6]:
# 1. Input: your own fields, on top of DeciderInput
class CodeReviewInput(DeciderInput):
    code: str
    language: str


# 2. What to ask about it
review_questions = {
    "is_bug_free": NoulQuestion(instructions="The code is free of bugs."),
    "security_risk": NoulQuestion(instructions="The code has a security vulnerability."),
    "severity": ChoiceQuestion(
        instructions="How severe is the worst problem in the code?",
        criteria={"low": "style or naming", "medium": "wrong results in some cases", "high": "data loss or a security hole"},
    ),
    "complexity": ScoreQuestion(
        instructions="How complex is the code?",
        criteria=["trivial", "simple", "moderate", "complex"],
    ),
}

# DANGEROUS: SQL Injection vulnerability
code_sample = """
def execute_query(user_input):
    query = f"SELECT * FROM users WHERE name = '{user_input}'"
    db.execute(query)
"""
review_input = CodeReviewInput(code=code_sample, language="python", questions=review_questions)

# 3. Same call as on the other backends; the result is a DeciderOutput
review = evaluator.evaluate(review_input).answers

print(f"bug free:      p={review['is_bug_free'].noul:.3f}")
print(f"security risk: p={review['security_risk'].noul:.3f}")
print(f"severity:      {review['severity'].choice}  confidence={review['severity'].confidence:.3f}  {review['severity'].probabilities}")
print(f"complexity:    {review['complexity'].score:.2f}  confidence={review['complexity'].confidence:.3f}  {review['complexity'].legend}")

bug free:      p=0.359
security risk: p=0.531
severity:      medium  confidence=0.268  {'low': 0.3019, 'medium': 0.5121, 'high': 0.186}
complexity:    1.40  confidence=0.443  {'0': 'trivial', '1': 'simple', '2': 'moderate', '3': 'complex'}


## Shaping the state

Subclass `DeciderInput` and your fields are sent as JSON, in the order you declare them. Override `build_state()` to render them as text instead, for example to put a rule in front of what is judged. Or pass `state` yourself.

Whatever the input class does, the request is checked against the model's contract before it is sent.

In [7]:
follows_rule = ChoiceQuestion(
    instructions="Does the text follow the rule?",
    criteria={"violates": "the text breaks the rule", "follows": "the text obeys the rule"},
)


# 1. Your own rendering of the state
class PortPolicy(DeciderInput):
    text: str

    def build_state(self):
        return f"Rule: services must only listen on port 443.\nText: {self.text}"


for text in ["The API endpoint defaults to port 8080.", "The service listens on 443."]:
    answer = evaluator.evaluate(PortPolicy(text=text, questions={"verdict": follows_rule})).answers["verdict"]
    print(answer.choice, text)

# 2. The state itself, as JSON
print(evaluator.evaluate(DeciderInput(
    state={"ticket": {"id": 7, "tier": "pro", "body": "Charged twice, fix it now."}},
    questions={"urgent": NoulQuestion(instructions="This needs a reply within the hour.")},
)).answers["urgent"].noul)

# Shorthand: with keywords, the Evaluator builds the DeciderInput for you
print(evaluator.evaluate(state="Charged twice, fix it now.", questions={"tone": TOXICITY}).answers["tone"].choice)

# 3. A request that breaks the contract never reaches the server
class Broken(DeciderInput):
    def build_state(self):
        return 42  # not text or JSON


error = evaluator.evaluate(Broken(questions={"verdict": follows_rule}))
print(error.error_code)

violates The API endpoint defaults to port 8080.
follows The service listens on 443.
0.4379
non-toxic
INVALID_REQUEST


In [8]:
# Stop the server warmup() started
evaluator.backend.shutdown()